In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")
trace_df = pd.read_csv("trace.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamp to UTC datetimes (new column 'ts')
for df in (metric_df, log_df, trace_df, error_df):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to build a compact summary row for a dataframe
def build_summary(df):
    total = len(df)
    if total == 0:
        min_ts = pd.NaT
        max_ts = pd.NaT
    else:
        min_ts = df['ts'].min()
        max_ts = df['ts'].max()
    return pd.DataFrame({
        'total_rows': [total],
        'min_ts_utc': [min_ts.isoformat() if not pd.isna(min_ts) else None],
        'max_ts_utc': [max_ts.isoformat() if not pd.isna(max_ts) else None]
    })

# Metric summaries
metric_summary = build_summary(metric_df)
metric_top_cmdb = metric_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
metric_top_kpi = metric_df['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='count').head(20)
metric_sample = metric_df.drop(columns=['ts'], errors='ignore').head(10)  # first 10 rows, show original columns

# Log summaries
log_summary = build_summary(log_df)
log_top_cmdb = log_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
if 'log_name' in log_df.columns:
    log_top_logname = log_df['log_name'].value_counts().rename_axis('log_name').reset_index(name='count').head(20)
else:
    log_top_logname = pd.DataFrame(columns=['log_name','count'])

# Trace summaries
trace_summary = build_summary(trace_df)
trace_top_cmdb = trace_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
if 'trace_name' in trace_df.columns:
    trace_top_tracename = trace_df['trace_name'].value_counts().rename_axis('trace_name').reset_index(name='count').head(20)
else:
    trace_top_tracename = pd.DataFrame(columns=['trace_name','count'])

# Error logs summaries
error_summary = build_summary(error_df)
error_top_cmdb = error_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
if 'message' in error_df.columns:
    error_top_messages = error_df['message'].value_counts().rename_axis('message_sample').reset_index(name='count').head(20)
else:
    error_top_messages = pd.DataFrame(columns=['message_sample','count'])

# Display compact summaries and top lists (limited to top 20), plus metric sample (10 rows)
metric_summary, metric_top_cmdb, metric_top_kpi, metric_sample, log_summary, log_top_cmdb, log_top_logname, trace_summary, trace_top_cmdb, trace_top_tracename, error_summary, error_top_cmdb, error_top_messages

```
Out[1]:
```
```python
# Reuse summary variables from earlier execution to build a concise plain-English summary.
summary = (
    "Data summary:\n\n"
    "metric.csv:\n"
    f"- total rows: {int(metric_summary.loc[0,'total_rows'])}\n"
    f"- time range (UTC): {metric_summary.loc[0,'min_ts_utc']} -> {metric_summary.loc[0,'max_ts_utc']}\n"
    "- top cmdb_id by row count (top shown):\n"
    f"{metric_top_cmdb.head(10).to_string(index=False)}\n"
    "- top kpi_name by count (top shown):\n"
    f"{metric_top_kpi.head(10).to_string(index=False)}\n"
    "- sample (first 10 rows):\n"
    f"{metric_sample.to_string(index=False)}\n\n"
    "log.csv:\n"
    f"- total rows: {int(log_summary.loc[0,'total_rows'])}\n"
    f"- time range (UTC): {log_summary.loc[0,'min_ts_utc']} -> {log_summary.loc[0,'max_ts_utc']}\n"
    "- top cmdb_id by row count (top shown):\n"
    f"{log_top_cmdb.head(10).to_string(index=False)}\n"
    "- top log_name by count:\n"
    f"{log_top_logname.to_string(index=False)}\n\n"
    "trace.csv:\n"
    f"- total rows: {int(trace_summary.loc[0,'total_rows'])}\n"
    "- no timestamps or trace_name entries present (empty dataset)\n\n"
    "error_logs.csv:\n"
    f"- total rows: {int(error_summary.loc[0,'total_rows'])}\n"
    "- no timestamps or error messages present (empty dataset)\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows                 min_ts_utc                 max_ts_utc
0        1855  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00,               cmdb_id  count
0           front-end    162
1              orders    162
2             payment    160
3           catalogue    150
4               carts    150
5                user    150
6            shipping    146
7           orders-db    100
8        catalogue-db    100
9            carts-db    100
10       queue-master    100
11         session-db    100
12           rabbitmq    100
13            user-db    100
14  rabbitmq-exporter     75,      kpi_name  count
0         cpu    375
1         mem    375
2      socket    375
3      diskio    189
4    workload    175
5  latency-50    171
6  latency-90    171
7       error     24,     timestamp   cmdb_id    kpi_name         value
0  1705769700     carts         cpu  1.833981e+00
1  1705769700     carts  latency-50  1.398532e-02
2  1705769700     carts  latency-90  2.382406e-02
3  1705769700     carts         mem  2.111587e+08
4  1705769700     carts      socket  9.317073e+00
5  1705769700     carts    workload  8.006341e+00
6  1705769700  carts-db         cpu  2.909910e+00
7  1705769700  carts-db      diskio  1.824100e+06
8  1705769700  carts-db         mem  7.547619e+07
9  1705769700  carts-db      socket  6.000000e+00,    total_rows                 min_ts_utc                 max_ts_utc
0         432  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00,          cmdb_id  count
0      catalogue     50
1         orders     50
2      front-end     50
3           user     50
4          carts     48
5       carts-db     48
6        payment     42
7   queue-master     42
8       shipping     42
9     session-db      8
10       user-db      2,           log_name  count
0  log.error_count    216
1    log.row_count    216,    total_rows min_ts_utc max_ts_utc
0           0       None       None, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [trace_name, count]
Index: [],    total_rows min_ts_utc max_ts_utc
0           0       None       None, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message_sample, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present; otherwise load.
try:
    metric_df
    log_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    log_df = pd.read_csv("log.csv")

# Ensure timestamp parsed to UTC datetimes (column 'ts')
for df in (metric_df, log_df):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    else:
        # ensure dtype
        df['ts'] = pd.to_datetime(df['ts'], utc=True)

# Ensure 'value' is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-20 16:52:19", tz="UTC")
window_end = pd.Timestamp("2024-01-20 17:22:19", tz="UTC")

# --- 1) metric.csv analysis ---

# Compute global P95 per (cmdb_id, kpi_name) using the entire metric_df (rule 9)
metric_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'global_p95'})
)

# Filter metric rows to incident window (inclusive)
metric_window = metric_df[(metric_df['ts'] >= window_start) & (metric_df['ts'] <= window_end)].copy()

# Prepare list to collect summary rows for every group in metric_p95
rows = []
for _, r in metric_p95.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    gp95 = r['global_p95']
    grp = metric_window[(metric_window['cmdb_id'] == cmdb) & (metric_window['kpi_name'] == kpi)]
    total_pts = int(len(grp))
    if total_pts == 0:
        anomaly_cnt = 0
        earliest = None
        latest = None
        max_val = None
    else:
        # anomaly: value >= global_p95
        anomalies = grp[grp['value'] >= gp95]
        anomaly_cnt = int(len(anomalies))
        if anomaly_cnt > 0:
            earliest = anomalies['ts'].min().isoformat()
            latest = anomalies['ts'].max().isoformat()
        else:
            earliest = None
            latest = None
        max_val = grp['value'].max()
    rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': float(gp95) if pd.notna(gp95) else np.nan,
        'total_points_in_window': total_pts,
        'anomaly_count': anomaly_cnt,
        'earliest_anomaly_ts_utc': earliest,
        'latest_anomaly_ts_utc': latest,
        'max_value_in_window': float(max_val) if pd.notna(max_val) else None
    })

metric_summary_df = pd.DataFrame(rows)

# Sort as requested and limit top 20
metric_summary_df = metric_summary_df.sort_values(
    by=['anomaly_count', 'max_value_in_window'], ascending=[False, False]
).reset_index(drop=True).head(20)

# --- 2) log.csv analysis ---

# Compute global P95 per (cmdb_id, log_name) using entire log_df
log_p95 = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'global_p95'})
)

# Filter log rows to incident window (inclusive)
log_window = log_df[(log_df['ts'] >= window_start) & (log_df['ts'] <= window_end)].copy()

# Build summary rows for logs (include groups even with zero points in window)
log_rows = []
for _, r in log_p95.iterrows():
    cmdb = r['cmdb_id']
    lname = r['log_name']
    gp95 = r['global_p95']
    grp = log_window[(log_window['cmdb_id'] == cmdb) & (log_window['log_name'] == lname)]
    total_pts = int(len(grp))
    if total_pts == 0:
        anomaly_cnt = 0
        earliest = None
        max_val = None
    else:
        anomalies = grp[grp['value'] >= gp95]
        anomaly_cnt = int(len(anomalies))
        earliest = anomalies['ts'].min().isoformat() if anomaly_cnt > 0 else None
        max_val = grp['value'].max()
    log_rows.append({
        'cmdb_id': cmdb,
        'log_name': lname,
        'global_p95': float(gp95) if pd.notna(gp95) else np.nan,
        'total_points_in_window': total_pts,
        'anomaly_count': anomaly_cnt,
        'earliest_anomaly_ts_utc': earliest,
        'max_value_in_window': float(max_val) if pd.notna(max_val) else None
    })

log_summary_df = pd.DataFrame(log_rows)
log_summary_df = log_summary_df.sort_values(by='anomaly_count', ascending=False).reset_index(drop=True).head(20)

# --- 3) distinct cmdb_id values present in metric anomalies ---
metric_anomaly_cmdbs = metric_summary_df.loc[metric_summary_df['anomaly_count'] > 0, 'cmdb_id'].drop_duplicates().reset_index(drop=True)
metric_anomaly_cmdb_count = int(metric_anomaly_cmdbs.nunique())

# Final displayed variables (compact)
metric_summary_df, log_summary_df, metric_anomaly_cmdbs, metric_anomaly_cmdb_count

```
Out[2]:
```
Summary of results (plain English):

Metric summary (top anomaly groups, sorted by anomaly_count then max value):
- rabbitmq-exporter · mem — global_p95=4,485,120.0; total points in window=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=4,485,120.0
- rabbitmq · socket — global_p95=11.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=11.0
- orders-db · socket — global_p95=7.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=7.0
- carts-db · socket — global_p95=6.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=6.0
- catalogue · socket — global_p95=5.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=5.0
- catalogue-db · socket — global_p95=4.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=4.0
- queue-master · socket — global_p95=3.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=3.0
- session-db · socket — global_p95=3.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=3.0
- rabbitmq-exporter · socket — global_p95=1.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=1.0
- catalogue-db · diskio — global_p95=0.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; latest=2024-01-20T17:19:00+00:00; max=0.0
(Additional metric summary rows were returned up to the top 20; several other mem/socket groups had smaller anomaly counts — e.g., user-db.socket had 14 anomalies, shipping.socket had 10, and some mem KPIs had 2–3 anomalies.)

Log summary (top groups):
- catalogue · log.error_count — global_p95=0.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; max=0.0
- front-end · log.error_count — global_p95=0.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; max=0.0
- orders · log.error_count — global_p95=0.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; max=0.0
- user · log.error_count — global_p95=0.0; total=25; anomaly_count=25; earliest=2024-01-20T16:55:00+00:00; max=0.0
- carts-db · log.row_count — global_p95=1.0; total=24; anomaly_count=24; earliest=2024-01-20T16:55:00+00:00; max=1.0
- carts · log.error_count — global_p95=0.0; total=24; anomaly_count=24; earliest=2024-01-20T16:55:00+00:00; max=0.0
- payment, shipping, queue-master, etc. also have log.error_count groups with anomaly_count=21–25 depending on available points.
(Other log row_count groups show spikes for front-end, orders, catalogue, queue-master with anomaly counts between 2–4 and higher max row_count values.)

Distinct cmdb_id values with metric anomalies:
- List of cmdb_id with anomaly_count > 0 (from metric summary): 
  rabbitmq-exporter, rabbitmq, orders-db, carts-db, catalogue, catalogue-db, queue-master, session-db, user-db, shipping, user, front-end, orders, carts
- Count of distinct cmdb_id with metric anomalies: 14

Notes:
- "global_p95" was computed on each group's full historical series before filtering to the incident window.
- Many socket and some mem/diskio groups exceed their group P95 throughout the incident window (several groups have anomaly_count equal to all points in-window). Several log.error_count groups have global_p95=0.0, which causes every non-null point (>=0) to be counted as an anomaly in this calculation.
- If you want, I can next: (a) display the full top-20 metric and log summaries as tables again, (b) inspect specific cmdb services in more detail, or (c) apply a different anomaly threshold logic (e.g., value > global_p95 instead of >=, or consider only >0 for log errors).

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name    global_p95  total_points_in_window  anomaly_count    earliest_anomaly_ts_utc      latest_anomaly_ts_utc  max_value_in_window
0   rabbitmq-exporter      mem  4.485120e+06                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         4.485120e+06
1            rabbitmq   socket  1.100000e+01                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         1.100000e+01
2           orders-db   socket  7.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         7.000000e+00
3            carts-db   socket  6.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         6.000000e+00
4           catalogue   socket  5.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         5.000000e+00
5        catalogue-db   socket  4.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         4.000000e+00
6        queue-master   socket  3.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         3.000000e+00
7          session-db   socket  3.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         3.000000e+00
8   rabbitmq-exporter   socket  1.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         1.000000e+00
9        catalogue-db   diskio  0.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         0.000000e+00
10           rabbitmq   diskio  0.000000e+00                      25             25  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00         0.000000e+00
11            user-db   socket  1.000000e+01                      25             14  2024-01-20T17:06:00+00:00  2024-01-20T17:19:00+00:00         1.000000e+01
12           shipping   socket  6.000000e+00                      25             10  2024-01-20T16:55:00+00:00  2024-01-20T17:08:00+00:00         6.000000e+00
13       queue-master      mem  2.654003e+08                      25              3  2024-01-20T17:15:00+00:00  2024-01-20T17:17:00+00:00         2.654016e+08
14               user   socket  1.900000e+01                      25              3  2024-01-20T17:17:00+00:00  2024-01-20T17:19:00+00:00         1.900000e+01
15          front-end   socket  7.233333e+00                      25              3  2024-01-20T16:57:00+00:00  2024-01-20T17:06:00+00:00         8.266667e+00
16       catalogue-db      mem  4.305215e+08                      25              2  2024-01-20T17:10:00+00:00  2024-01-20T17:12:00+00:00         4.305743e+08
17             orders      mem  3.333748e+08                      25              2  2024-01-20T17:18:00+00:00  2024-01-20T17:19:00+00:00         3.335133e+08
18           shipping      mem  2.956465e+08                      25              2  2024-01-20T17:18:00+00:00  2024-01-20T17:19:00+00:00         2.958520e+08
19              carts      mem  2.145781e+08                      25              2  2024-01-20T17:18:00+00:00  2024-01-20T17:19:00+00:00         2.146701e+08,          cmdb_id         log_name  global_p95  total_points_in_window  anomaly_count    earliest_anomaly_ts_utc  max_value_in_window
0      catalogue  log.error_count         0.0                      25             25  2024-01-20T16:55:00+00:00                  0.0
1      front-end  log.error_count         0.0                      25             25  2024-01-20T16:55:00+00:00                  0.0
2         orders  log.error_count         0.0                      25             25  2024-01-20T16:55:00+00:00                  0.0
3           user  log.error_count         0.0                      25             25  2024-01-20T16:55:00+00:00                  0.0
4       carts-db    log.row_count         1.0                      24             24  2024-01-20T16:55:00+00:00                  1.0
5          carts  log.error_count         0.0                      24             24  2024-01-20T16:55:00+00:00                  0.0
6       carts-db  log.error_count         0.0                      24             24  2024-01-20T16:55:00+00:00                  0.0
7        payment  log.error_count         0.0                      21             21  2024-01-20T16:55:00+00:00                  0.0
8       shipping  log.error_count         0.0                      21             21  2024-01-20T16:55:00+00:00                  0.0
9   queue-master  log.error_count         0.0                      21             21  2024-01-20T16:55:00+00:00                  0.0
10    session-db  log.error_count         0.0                       4              4  2024-01-20T16:59:00+00:00                  0.0
11         carts    log.row_count       125.0                      24              4  2024-01-20T16:56:00+00:00                150.0
12    session-db    log.row_count         5.0                       4              4  2024-01-20T16:59:00+00:00                  5.0
13      shipping    log.row_count        64.0                      21              3  2024-01-20T16:58:00+00:00                 65.0
14  queue-master    log.row_count       576.0                      21              3  2024-01-20T16:58:00+00:00                585.0
15     catalogue    log.row_count       166.2                      25              2  2024-01-20T17:03:00+00:00                168.0
16     front-end    log.row_count      2035.0                      25              2  2024-01-20T17:03:00+00:00               2045.0
17        orders    log.row_count      1836.0                      25              2  2024-01-20T17:10:00+00:00               1860.0
18       payment    log.row_count       104.0                      21              2  2024-01-20T16:58:00+00:00                105.0
19          user    log.row_count       608.4                      25              2  2024-01-20T16:58:00+00:00                621.0, 0     rabbitmq-exporter
1              rabbitmq
2             orders-db
3              carts-db
4             catalogue
5          catalogue-db
6          queue-master
7            session-db
8               user-db
9              shipping
10                 user
11            front-end
12               orders
13                carts
Name: cmdb_id, dtype: object, 14)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing metric_df and log_df; ensure ts and numeric value
for df in (metric_df, log_df):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    else:
        df['ts'] = pd.to_datetime(df['ts'], utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Incident window (UTC)
window_start = pd.Timestamp("2024-01-20 16:52:19", tz="UTC")
window_end = pd.Timestamp("2024-01-20 17:22:19", tz="UTC")

# 1) Metric: catalogue kpi segments

# Compute global P95 per kpi_name for cmdb_id == 'catalogue' using full series (before filtering)
catalogue_metric_all = metric_df[metric_df['cmdb_id'] == 'catalogue'].copy()
metric_p95_kpis = (
    catalogue_metric_all
    .groupby('kpi_name', dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Filter catalogue rows to incident window (sorted)
catalogue_window = metric_df[
    (metric_df['cmdb_id'] == 'catalogue') &
    (metric_df['ts'] >= window_start) &
    (metric_df['ts'] <= window_end)
].copy().sort_values('ts')

# Map global_p95 into the windowed rows
catalogue_window = catalogue_window.merge(metric_p95_kpis, on='kpi_name', how='left')

# Identify consecutive anomalous segments per kpi (value >= global_p95)
segments = []
for kpi, grp in catalogue_window.groupby('kpi_name', sort=False):
    g = grp.sort_values('ts').reset_index(drop=True)
    # If no p95 (NaN) skip
    gp95 = metric_p95_kpis.loc[metric_p95_kpis['kpi_name'] == kpi, 'global_p95'].iloc[0]
    # boolean anomaly flag
    g['is_anom'] = g['value'] >= gp95
    if not g['is_anom'].any():
        continue
    # create segment ids where anomaly state changes
    g['seg_change'] = (g['is_anom'] != g['is_anom'].shift(fill_value=False)).cumsum()
    # keep only anomalous segments
    anom_groups = g[g['is_anom']].groupby('seg_change')
    for seg_id, seg in anom_groups:
        seg_start = seg['ts'].min()
        seg_end = seg['ts'].max()
        seg_len = int(len(seg))
        seg_max = float(seg['value'].max()) if seg_len > 0 else None
        # severity ratio handling when gp95 is zero
        if pd.isna(gp95):
            severity = np.nan
        else:
            if gp95 == 0:
                severity = np.inf if seg_max and seg_max > 0 else (1.0 if seg_max == 0 else np.nan)
            else:
                severity = seg_max / float(gp95)
        segments.append({
            'kpi_name': kpi,
            'global_p95': float(gp95) if pd.notna(gp95) else np.nan,
            'segment_start_ts_utc': seg_start.isoformat(),
            'segment_end_ts_utc': seg_end.isoformat(),
            'segment_length_points': seg_len,
            'max_value_in_segment': seg_max,
            'severity_ratio': float(severity) if np.isfinite(severity) or (severity == np.inf) else (None if pd.isna(severity) else severity)
        })

catalogue_segments_df = pd.DataFrame(segments)

# Sort and limit top 20 as requested
if not catalogue_segments_df.empty:
    catalogue_segments_df = catalogue_segments_df.sort_values(
        by=['segment_length_points', 'severity_ratio'],
        ascending=[False, False]
    ).reset_index(drop=True).head(20)

# 2) Logs: catalogue rows in window and per-log_name p95+anomaly counts

# Compute global P95 per log_name for cmdb_id == 'catalogue' using full series
catalogue_log_all = log_df[log_df['cmdb_id'] == 'catalogue'].copy()
log_p95_kpis = (
    catalogue_log_all
    .groupby('log_name', dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Filter catalogue logs to incident window
catalogue_log_window = log_df[
    (log_df['cmdb_id'] == 'catalogue') &
    (log_df['ts'] >= window_start) &
    (log_df['ts'] <= window_end)
].copy().sort_values('ts')

# Prepare rows display: timestamp_utc, log_name, value (limit to 20)
if not catalogue_log_window.empty:
    catalogue_log_rows_df = catalogue_log_window[['ts', 'log_name', 'value']].copy()
    catalogue_log_rows_df['timestamp_utc'] = catalogue_log_rows_df['ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    catalogue_log_rows_df = catalogue_log_rows_df[['timestamp_utc', 'log_name', 'value']].head(20)
else:
    catalogue_log_rows_df = pd.DataFrame(columns=['timestamp_utc', 'log_name', 'value'])

# Compute anomaly_count per log_name in-window using the group global_p95
if not log_p95_kpis.empty:
    # merge p95 into window rows
    log_win_with_p95 = catalogue_log_window.merge(log_p95_kpis, on='log_name', how='left')
    # anomaly flag
    log_win_with_p95['is_anom'] = log_win_with_p95['value'] >= log_win_with_p95['global_p95']
    log_summary = (
        log_win_with_p95
        .groupby('log_name', sort=False)
        .agg(global_p95=('global_p95', 'first'),
             anomaly_count_in_window=('is_anom', 'sum'),
             total_points_in_window=('value', 'count'))
        .reset_index()
    )
    # ensure types
    log_summary['global_p95'] = log_summary['global_p95'].astype(float)
    log_summary['anomaly_count_in_window'] = log_summary['anomaly_count_in_window'].astype(int)
    catalogue_log_summary_df = log_summary[['log_name', 'global_p95', 'anomaly_count_in_window']].head(20)
else:
    catalogue_log_summary_df = pd.DataFrame(columns=['log_name', 'global_p95', 'anomaly_count_in_window'])

# Display compact results (stored in kernel variables)
catalogue_segments_df, catalogue_log_rows_df, catalogue_log_summary_df, metric_p95_kpis, log_p95_kpis

```
Out[3]:
```
Catalogue metric anomaly segments (top segments, sorted by length then severity):

- socket
  - global_p95: 5.0
  - segment: 2024-01-20T16:55:00+00:00 → 2024-01-20T17:19:00+00:00
  - length: 25 points (anomalous for the whole incident window)
  - max_value_in_segment: 5.0
  - severity_ratio: 1.00

- mem
  - global_p95: 6,099,053.0
  - segment: 2024-01-20T17:18:00+00:00 → 2024-01-20T17:19:00+00:00
  - length: 2 points
  - max_value_in_segment: 6,107,136.0
  - severity_ratio: 1.0013

- cpu
  - global_p95: 0.1863477
  - segment: 2024-01-20T17:03:00+00:00 → same
  - length: 1 point
  - max_value_in_segment: 0.1991669
  - severity_ratio: 1.0688

- latency-90
  - global_p95: 0.004747993
  - two 1-point segments at 2024-01-20T17:01:00+00:00 and 2024-01-20T17:18:00+00:00
  - each length: 1 point
  - max values: ~0.0048565 and ~0.00475307
  - severity_ratios: ~1.023 and ~1.0011

- latency-50
  - global_p95: 0.003082218
  - two 1-point segments at 2024-01-20T17:01:00+00:00 and 2024-01-20T17:18:00+00:00
  - max values: ~0.0031425 and ~0.0030850
  - severity_ratios: ~1.0196 and ~1.0009

- workload
  - global_p95: 4.237283
  - two separate 1-point segments at 2024-01-20T17:03:00 and 2024-01-20T17:19:00
  - max values: ~4.2566 and ~4.2603
  - severity_ratios: ~1.0046 and ~1.0054

(These are the top segments found; socket is the dominant continuous anomaly across the full incident window for catalogue.)

Catalogue log rows in the incident window (sample up to 20 rows shown earlier):
- The logs alternate between log.error_count and log.row_count entries every minute in the window. Examples include:
  - 2024-01-20T16:55:00+00:00 — log.error_count = 0
  - 2024-01-20T16:55:00+00:00 — log.row_count = 107
  - 2024-01-20T16:56:00+00:00 — log.error_count = 0
  - 2024-01-20T16:56:00+00:00 — log.row_count = 161
  - ... (continued per-minute pairs through the window)

Per-log global P95 and anomaly counts (computed on full series, then counted in-window):
- log.error_count
  - global_p95 = 0.0
  - anomaly_count in window = 25 (every point meets value >= 0.0)
- log.row_count
  - global_p95 = 166.2
  - anomaly_count in window = 2 (two in-window row_count points >= 166.2)

Additional notes:
- Global P95 values for catalogue KPIs (computed on the full historical series) used here: cpu ~0.1863, latency-50 ~0.003082, latency-90 ~0.004748, mem ~6,099,053, socket = 5.0, workload ~4.2373.
- The most significant finding: catalogue.socket is anomalous for the entire incident window (25/25 points). Other KPIs show brief, single- or two-point spikes near the end of the window with severity ratios only slightly above 1.0. Logs show no error spikes (error_count is zero throughout but counted as anomalous because global_p95=0.0); row_count exceeded its p95 only twice.

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name    global_p95       segment_start_ts_utc         segment_end_ts_utc  segment_length_points  max_value_in_segment  severity_ratio
0      socket  5.000000e+00  2024-01-20T16:55:00+00:00  2024-01-20T17:19:00+00:00                     25          5.000000e+00        1.000000
1         mem  6.099053e+06  2024-01-20T17:18:00+00:00  2024-01-20T17:19:00+00:00                      2          6.107136e+06        1.001325
2         cpu  1.863477e-01  2024-01-20T17:03:00+00:00  2024-01-20T17:03:00+00:00                      1          1.991669e-01        1.068792
3  latency-90  4.747993e-03  2024-01-20T17:01:00+00:00  2024-01-20T17:01:00+00:00                      1          4.856504e-03        1.022854
4  latency-50  3.082218e-03  2024-01-20T17:01:00+00:00  2024-01-20T17:01:00+00:00                      1          3.142502e-03        1.019559
5    workload  4.237283e+00  2024-01-20T17:19:00+00:00  2024-01-20T17:19:00+00:00                      1          4.260250e+00        1.005420
6    workload  4.237283e+00  2024-01-20T17:03:00+00:00  2024-01-20T17:03:00+00:00                      1          4.256633e+00        1.004567
7         cpu  1.863477e-01  2024-01-20T17:01:00+00:00  2024-01-20T17:01:00+00:00                      1          1.867522e-01        1.002171
8  latency-90  4.747993e-03  2024-01-20T17:18:00+00:00  2024-01-20T17:18:00+00:00                      1          4.753070e-03        1.001069
9  latency-50  3.082218e-03  2024-01-20T17:18:00+00:00  2024-01-20T17:18:00+00:00                      1          3.085039e-03        1.000915,                 timestamp_utc         log_name  value
4    2024-01-20T16:55:00+0000  log.error_count      0
5    2024-01-20T16:55:00+0000    log.row_count    107
22   2024-01-20T16:56:00+0000  log.error_count      0
23   2024-01-20T16:56:00+0000    log.row_count    161
40   2024-01-20T16:57:00+0000  log.error_count      0
41   2024-01-20T16:57:00+0000    log.row_count    160
58   2024-01-20T16:58:00+0000  log.error_count      0
59   2024-01-20T16:58:00+0000    log.row_count    161
76   2024-01-20T16:59:00+0000  log.error_count      0
77   2024-01-20T16:59:00+0000    log.row_count    163
96   2024-01-20T17:00:00+0000  log.error_count      0
97   2024-01-20T17:00:00+0000    log.row_count    159
113  2024-01-20T17:01:00+0000    log.row_count    162
112  2024-01-20T17:01:00+0000  log.error_count      0
130  2024-01-20T17:02:00+0000  log.error_count      0
131  2024-01-20T17:02:00+0000    log.row_count    162
148  2024-01-20T17:03:00+0000  log.error_count      0
149  2024-01-20T17:03:00+0000    log.row_count    167
166  2024-01-20T17:04:00+0000  log.error_count      0
167  2024-01-20T17:04:00+0000    log.row_count    151,           log_name  global_p95  anomaly_count_in_window
0  log.error_count         0.0                       25
1    log.row_count       166.2                        2,      kpi_name    global_p95
0         cpu  1.863477e-01
1  latency-50  3.082218e-03
2  latency-90  4.747993e-03
3         mem  6.099053e+06
4      socket  5.000000e+00
5    workload  4.237283e+00,           log_name  global_p95
0  log.error_count         0.0
1    log.row_count       166.2)```
```